In [1]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

csv_path = os.path.join(path, "IMDB Dataset.csv")

df = pd.read_csv(csv_path)
display(df.head())

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [41]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [2]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [3]:
df.shape

(50000, 2)

In [4]:
corpus = []
stop_words = set(stopwords.words('english'))
for i in range(0,df.shape[0]):
  update = re.sub('[^a-zA-Z0-9]',' ',df['review'][i])
  update = update.lower()
  update = update.split()

  update = [lemmatizer.lemmatize(word) for word in update if not word in stop_words]

  update = " ".join(update)

  corpus.append(update  )

In [5]:
len(corpus)

50000

In [6]:
!pip install gensim
import gensim
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
nltk.download('punkt')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 68.0 MB/s  0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [7]:
words = [simple_preprocess(word) for word in corpus]

In [8]:
len(words)

50000

In [9]:
model = gensim.models.Word2Vec(words, window=10, min_count=2, vector_size=300)

In [10]:
import numpy as np

def document_vector(word_list, model):
    # Filter out words that are not in the Word2Vec vocabulary
    words_in_vocab = [word for word in word_list if word in model.wv]
    if len(words_in_vocab) == 0:
        return np.zeros(model.vector_size)
    # Return the mean vector
    return np.mean(model.wv[words_in_vocab], axis=0)

# Generate feature matrix X

# regular expression -> stopwords -> lemmatize -> only embedded words
X = np.array([document_vector(review, model) for review in words])
print("Feature matrix shape:", X.shape)

Feature matrix shape: (50000, 300)


In [11]:
# Use pd.get_dummies to encode the target variable
y_dummies = pd.get_dummies(df['sentiment'], drop_first=True)
# Convert to a 1D array
y = y_dummies.values.ravel()
print("Target shape:", y.shape)

Target shape: (50000,)


In [12]:
from sklearn.model_selection import train_test_split

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (40000, 300)
Test size: (10000, 300)


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# Make predictions and evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8507
              precision    recall  f1-score   support

       False       0.86      0.84      0.85      5000
        True       0.84      0.86      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [14]:
test = "Its just awesome, Its been a very long time since i have watched this."
test2 = "Its just aweful,poor story , weaker plots and pure time waste."

In [18]:
def preprocess_embed(text):
  print(text,"\n")
  update = re.sub('[^a-zA-Z0-9]',' ',text)
  update = update.lower()
  update = update.split()

  update = [lemmatizer.lemmatize(word) for word in update if not word in stop_words]
  update = " ".join(update)

  update  = simple_preprocess(update)
  update = document_vector(update, model)

  return update

In [21]:
def predict(text,prob):
  clean_text = preprocess_embed(text)
  res = "Positive" if bool(clf.predict([clean_text])[0]) else "Negative"
  if(prob):
    proba = clf.predict_proba([clean_text])
    print(proba)
  return res

In [26]:
predict(test , prob=True)

Its just awesome, Its been a very long time since i have watched this. 

[[0.45 0.55]]


'Positive'

In [25]:
predict(test2 , prob=True)

Its just aweful,poor story , weaker plots and pure time waste. 

[[0.73 0.27]]


'Negative'

In [27]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train,y_train)
lr.score(X_test,y_test)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8787

In [31]:
vec = preprocess_embed(test)

Its just awesome, Its been a very long time since i have watched this. 



In [33]:
lr.predict([vec])

array([ True])

In [38]:
lr_y_pred = lr.predict(X_test)

clf_y_pred = clf.predict(X_test)

In [39]:
print(classification_report(lr_y_pred,y_test))

              precision    recall  f1-score   support

       False       0.88      0.88      0.88      5003
        True       0.88      0.88      0.88      4997

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [40]:
print(classification_report(clf_y_pred,y_test))

              precision    recall  f1-score   support

       False       0.84      0.86      0.85      4877
        True       0.86      0.84      0.85      5123

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

